In [ ]:
# reload packages
%load_ext autoreload
%autoreload 2

### Choose GPU (this may not be needed on your computer)

In [ ]:
%env CUDA_DEVICE_ORDER=PCI_BUS_ID
%env CUDA_VISIBLE_DEVICES=''

### load packages

In [ ]:
from tfumap.umap import tfUMAP

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tqdm.autonotebook import tqdm
import umap
import pandas as pd

### Load dataset

In [ ]:
from tfumap.paths import ensure_dir, MODEL_DIR, DATA_DIR

In [ ]:
syllable_df = pd.read_pickle(DATA_DIR/'cassins'/ 'cassins.pickle')

In [ ]:
syllable_df[:3]

In [ ]:
top_labels = (
    pd.DataFrame(
        {i: [np.sum(syllable_df.labels.values == i)] for i in syllable_df.labels.unique()}
    )
    .T.sort_values(by=0, ascending=False)[:20]
    .T
)
top_labels

In [ ]:
sylllable_df = syllable_df[syllable_df.labels.isin(top_labels.columns)]

In [ ]:
sylllable_df[:3]

In [ ]:
sylllable_df = sylllable_df.reset_index()

In [ ]:
specs = np.array(list(sylllable_df.spectrogram.values))
specs.shape

In [ ]:
sylllable_df['subset'] = 'train'
sylllable_df.loc[:1000, 'subset'] = 'valid'
sylllable_df.loc[1000:1999, 'subset'] = 'test'

In [ ]:
len(sylllable_df)

In [ ]:
Y_train = np.array(list(sylllable_df.labels.values[sylllable_df.subset == 'train']))
Y_valid = np.array(list(sylllable_df.labels.values[sylllable_df.subset == 'valid']))
Y_test = np.array(list(sylllable_df.labels.values[sylllable_df.subset == 'test']))

In [ ]:
X_train = np.array(list(sylllable_df.spectrogram.values[sylllable_df.subset == 'train'])) #/ 255.
X_valid = np.array(list(sylllable_df.spectrogram.values[sylllable_df.subset == 'valid']))# / 255.
X_test = np.array(list(sylllable_df.spectrogram.values[sylllable_df.subset == 'test'])) #/ 255.

In [ ]:
X_train_flat = X_train.reshape((len(X_train), np.product(np.shape(X_train)[1:])))

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder()
Y_train = enc.fit_transform([[i] for i in Y_train]).astype('int').flatten()

In [ ]:
plt.matshow(X_train[10])

### Create model and train

In [ ]:
embedder = tfUMAP(n_components = 64, direct_embedding=True, verbose=True, negative_sample_rate=5, training_epochs=100)

In [ ]:
z = embedder.fit_transform(X_train_flat)

### Plot model output

In [ ]:
fig, ax = plt.subplots( figsize=(8, 8))
sc = ax.scatter(
    z[:, 0],
    z[:, 1],
    c=Y_train[:len(z)],
    cmap="tab20",
    s=0.1,
    alpha=0.5,
    rasterized=True,
)
ax.axis('equal')
ax.set_title("UMAP in Tensorflow embedding", fontsize=20)
plt.colorbar(sc, ax=ax);

### View loss

In [ ]:
from tfumap.umap import retrieve_tensors
import seaborn as sns

In [ ]:
loss_df = retrieve_tensors(embedder.tensorboard_logdir)
loss_df[:3]

In [ ]:
ax = sns.lineplot(x="step", y="val", hue="group", data=loss_df[loss_df.variable=='umap_loss'])
ax.set_xscale('log')

### Compare to direct embedding with base UMAP

In [ ]:
from umap import UMAP

In [ ]:
z_umap = UMAP(metric = dtw_metric, verbose=True, n_components = 64).fit_transform(X_train_flat)

In [ ]:
### Plot model output

fig, ax = plt.subplots( figsize=(8, 8))
sc = ax.scatter(
    z_umap[:, 0],
    z_umap[:, 1],
    c=Y_train[:len(z)],
    cmap="tab20",
    s=0.1,
    alpha=0.5,
    rasterized=True,
)
ax.axis('equal')
ax.set_title("UMAP in Tensorflow embedding", fontsize=20)
plt.colorbar(sc, ax=ax);

### Save output

In [ ]:
import os
import pickle

In [ ]:
from tfumap.paths import ensure_dir, MODEL_DIR

In [ ]:
output_dir = MODEL_DIR/'projections'/ 'cassins_dtw' / '64' / 'umap-learn'
ensure_dir(output_dir)

In [ ]:
with open(os.path.join(output_dir, "model.pkl"), "wb") as output:
    pickle.dump(embedder, output, pickle.HIGHEST_PROTOCOL)

In [ ]:
np.save(output_dir / 'z.npy', z_umap)